In [ ]:
# FLORES-200 翻译评估

本地 vLLM + FLORES-200 数据集评估翻译模型 BLEU。  
脚本：`flores200_single`（单语对）、`flores200_multilingual`（7 语种 42 向）、`run_evaluation`（可选自动起 vLLM）。

---

## 1. 一键运行（自动启动 vLLM 再评估）

```bash
# 7 语种 42 向评估，每语对 50 条试跑
python run_evaluation.py --mode multilingual --limit 50

# 单语对（英→中）
python run_evaluation.py --mode single --config eng_Latn-zho_Hans --limit 100
```

---

## 2. 分步运行（先起 vLLM，再跑评估）

**终端 1 — 启动 vLLM（端口 8005）：**
```bash
source .venv/bin/activate
NVIDIA_VISIBLE_DEVICES=0,1,2,3 vllm serve ./huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a/  --port 8005   --tensor-parallel-size 4  --served-model-name Qwen/Qwen3.5-9B

NVIDIA_VISIBLE_DEVICES=0,1,2,3 vllm serve ./huggingface/hub/models--Qwen--Qwen3.5-9B/snapshots/c202236235762e1c871ad0ccb60c8ee5ba337b9a/ --port 8005 --tensor-parallel-size 1 --max-model-len 262144 --language-model-only --served-model-name Qwen/Qwen3.5-9B

```

**终端 2 — 只跑评估（加 `--no-serve`）：**
```bash
source .venv/bin/activate
python run_evaluation.py --no-serve --mode multilingual --limit 50
# 或单语对
python run_evaluation.py --no-serve --mode single --config eng_Latn-zho_Hans
```

---

## 3. 直接调用评估脚本

API 默认：`http://localhost:8005/v1`

| 用途 | 命令 |
|------|------|
| 单语对 | `python flores200_single.py [--config eng_Latn-zho_Hans] [--limit N] [--output out.json]` |
| 多语种 42 向 | `python flores200_multilingual.py [--limit N] [--output-dir dir] [--summary summary.json]` |

多语种脚本内部调用 `flores200_single.run_single_evaluation()`。

---

## 4. 可选参数（run_evaluation.py）

| 参数 | 说明 |
|------|------|
| `--no-serve` | 不启动 vLLM，只跑评估 |
| `--gpus 4,5` | 指定 GPU（不填会交互询问） |
| `--port 8005` | vLLM 端口 |
| `--model` | 模型名或本地路径 |
| `--limit N` | 每语对只评估前 N 条（试跑用） |